# S5 Bangla — post-generation scoring and analysis (Kaggle)

Run this **only after all three generation replicates are complete**. Attach exactly one final S5 checkpoint dataset containing `s5_main_bn_cases.jsonl` (5,400 rows) and the sealed `thesis-verifier-b` dataset. This notebook does not load Gemma-3, call Google, or generate anything. It scores final/loop-attempt texts with Verifier-B, writes a separate score archive, then produces the pre-specified paired bootstrap/BH tables.

In [ ]:
from pathlib import Path
import os, subprocess
POSTRUN_COMMIT = '81675ff0e273b1508912a1db63eadebcf1a324f6'
REPO = Path('/kaggle/working/s5_postrun_81675ff')
if not (REPO / '.git').is_dir():
    subprocess.run(['git','clone','-q','https://github.com/alphapie77/BSc_Thesis.git',str(REPO)], check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',POSTRUN_COMMIT], check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip() == POSTRUN_COMMIT
os.chdir(REPO)
subprocess.run(['git','log','--oneline','-1'], check=True)


In [ ]:
# Fresh subprocesses avoid stale imports; no kernel restart is needed.
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','uninstall','-y','sklearn-compat'], check=False)
subprocess.run([sys.executable,'-m','pip','install','-q','-U','scikit-learn==1.9.0','transformers==5.15.0','pyyaml','joblib','nltk','pytest'], check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_s5_postrun.py','-q'], check=True)


In [ ]:
# Restore the final generation archive and locate the sealed Verifier-B.
from pathlib import Path
import glob, shutil
REPO = Path('/kaggle/working/s5_postrun_81675ff')
os.chdir(REPO)
cases = glob.glob('/kaggle/input/**/s5_main_bn_cases.jsonl', recursive=True)
assert len(cases) == 1, f'attach exactly one FINAL checkpoint dataset; found {cases}'
dst = Path('results/s5_main_bn_cases.jsonl'); dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(cases[0], dst)
# The frozen calibration artifact is versioned in this exact repo commit.
# The attached thesis-verifier-b dataset supplies only its large HF weights.
VERIFIER_B_ARTIFACT = str(Path('artifacts/verifier_b.joblib'))
assert Path(VERIFIER_B_ARTIFACT).is_file(), 'registered verifier-B artifact missing from repo'
weights = [str(Path(p).parent) for p in glob.glob('/kaggle/input/**/model.safetensors', recursive=True)]
assert len(weights) == 1, f'attach exactly one thesis-verifier-b weights dataset; found {weights}'
VERIFIER_B_WEIGHTS = weights[0]
print('restored final cases:', dst.stat().st_size, 'bytes')
print('Verifier-B artifact and weights: present (paths not printed)')


In [ ]:
# This refuses anything except the complete 90×2×10×3 Bangla surface.
import subprocess, sys
subprocess.run([sys.executable,'-m','src.eval.score_s5_bn','--config','configs/s5_score_bn.yaml','--verifier-b-path',VERIFIER_B_ARTIFACT,'--verifier-b-weights',VERIFIER_B_WEIGHTS,'--device','cuda'], check=True)
subprocess.run([sys.executable,'-m','src.eval.analyze_s5_bn','--config','configs/s5_analysis_bn.yaml'], check=True)
subprocess.run([sys.executable,'-m','src.eval.analyze_s5_goodhart_bn','--config','configs/s5_goodhart_bn.yaml'], check=True)
subprocess.run([sys.executable,'-m','src.eval.analyze_s5_diversity_realism_bn','--config','configs/s5_diversity_realism_bn.yaml'], check=True)


In [ ]:
# Save this results bundle as a Kaggle dataset and also download it.
from pathlib import Path
import shutil
bundle = Path('/kaggle/working/s5_bn_postrun_results')
if bundle.is_dir(): shutil.rmtree(bundle)
bundle.mkdir()
for name in ['s5_main_bn_cases.jsonl','s5_main_bn_verifier_b_scores.jsonl','s5_main_bn_scored_cases.csv','s5_main_bn_score_manifest.json','s5_main_bn_master_table.csv','s5_main_bn_paired_statistics.csv','s5_main_bn_goodhart_by_attempt.csv','s5_main_bn_goodhart_attempt_summary.csv','s5_main_bn_goodhart_paired_transitions.csv','s5_main_bn_goodhart_report.json','s5_main_bn_diversity.csv','s5_main_bn_length_js.csv','s5_main_bn_realism_preflight.json','s5_main_bn_analysis.json']:
    src = Path('results') / name
    assert src.is_file(), f'missing expected result: {src}'
    shutil.copy2(src, bundle / name)
archive = shutil.make_archive('/kaggle/working/s5_bn_postrun_results','zip',bundle)
print('DOWNLOAD OR SAVE AS DATASET:', archive)
